<a href="https://colab.research.google.com/github/Dharshini1701/priyadharshini/blob/main/gemini_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install -q -U google-genai

import os
from getpass import getpass
from google import genai

api_key = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=api_key)

print("SUCCESS - Gemini client created!")

Enter your Gemini API key: ··········
SUCCESS - Gemini client created!


In [1]:
import sqlite3
import re
import time

# Create database
conn = sqlite3.connect("sales_nl2sql.db")
cursor = conn.cursor()

# Create table
cursor.execute("""
CREATE TABLE IF NOT EXISTS sales (
    SaleID INTEGER PRIMARY KEY,
    ProductName TEXT,
    Category TEXT,
    City TEXT,
    Quantity INTEGER,
    Amount REAL,
    SaleDate TEXT,
    Status TEXT
)
""")


In [2]:
sales_data = [
    (1, "Laptop", "Electronics", "Chennai", 2, 120000, "2026-01-10", "Completed"),
    (2, "Mobile", "Electronics", "Coimbatore", 3, 75000, "2026-01-12", "Completed"),
    (3, "Headphones", "Accessories", "Bengaluru", 5, 25000, "2026-01-15", "Completed"),
    (4, "Tablet", "Electronics", "Chennai", 2, 60000, "2026-01-20", "Cancelled"),
    (5, "Keyboard", "Accessories", "Madurai", 4, 12000, "2026-01-22", "Completed"),
    (6, "Monitor", "Electronics", "Coimbatore", 3, 45000, "2026-02-02", "Completed"),
    (7, "Mouse", "Accessories", "Chennai", 10, 10000, "2026-02-05", "Completed"),
    (8, "Printer", "Electronics", "Bengaluru", 2, 30000, "2026-02-10", "Completed"),
    (9, "Laptop", "Electronics", "Madurai", 1, 65000, "2026-02-15", "Completed"),
    (10, "Mobile", "Electronics", "Chennai", 4, 100000, "2026-02-20", "Completed")
]

# Insert data
cursor.executemany("""
INSERT OR IGNORE INTO sales
(SaleID, ProductName, Category, City, Quantity, Amount, SaleDate, Status)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", sales_data)

conn.commit()

print("10 records inserted successfully!")

10 records inserted successfully!


In [3]:
# Display records
print("DATABASE RECORDS")
print("-" * 60)

cursor.execute("SELECT * FROM sales")
rows = cursor.fetchall()

for row in rows:
    print(row)

# Get schema
cursor.execute("PRAGMA table_info(sales)")
schema_rows = cursor.fetchall()

schema = """
TABLE: sales
COLUMNS:
"""

for row in schema_rows:
    column_name = row[1]
    data_type = row[2]
    schema += f"{column_name} {data_type}\n"

print("\nDATABASE SCHEMA")
print("-" * 60)
print(schema)

DATABASE RECORDS
------------------------------------------------------------
(1, 'Laptop', 'Electronics', 'Chennai', 2, 120000.0, '2026-01-10', 'Completed')
(2, 'Mobile', 'Electronics', 'Coimbatore', 3, 75000.0, '2026-01-12', 'Completed')
(3, 'Headphones', 'Accessories', 'Bengaluru', 5, 25000.0, '2026-01-15', 'Completed')
(4, 'Tablet', 'Electronics', 'Chennai', 2, 60000.0, '2026-01-20', 'Cancelled')
(5, 'Keyboard', 'Accessories', 'Madurai', 4, 12000.0, '2026-01-22', 'Completed')
(6, 'Monitor', 'Electronics', 'Coimbatore', 3, 45000.0, '2026-02-02', 'Completed')
(7, 'Mouse', 'Accessories', 'Chennai', 10, 10000.0, '2026-02-05', 'Completed')
(8, 'Printer', 'Electronics', 'Bengaluru', 2, 30000.0, '2026-02-10', 'Completed')
(9, 'Laptop', 'Electronics', 'Madurai', 1, 65000.0, '2026-02-15', 'Completed')
(10, 'Mobile', 'Electronics', 'Chennai', 4, 100000.0, '2026-02-20', 'Completed')

DATABASE SCHEMA
------------------------------------------------------------

TABLE: sales
COLUMNS:
SaleID INT

In [4]:
def validate_sql(sql):
    sql = sql.strip()
    sql_upper = sql.upper()

    # Only SELECT
    if not sql_upper.startswith("SELECT"):
        return False

    # Reject multiple statements
    if ";" in sql[:-1]:
        return False

    # Dangerous commands
    dangerous_commands = [
        "INSERT",
        "UPDATE",
        "DELETE",
        "DROP",
        "ALTER",
        "CREATE",
        "REPLACE",
        "ATTACH",
        "DETACH"
    ]

    for command in dangerous_commands:
        if re.search(rf"\b{command}\b", sql_upper):
            return False

    return True

In [5]:
def clean_sql(sql):
    sql = sql.strip()

    # Remove markdown code blocks
    sql = re.sub(
        r"```sql",
        "",
        sql,
        flags=re.IGNORECASE
    )

    sql = re.sub(
        r"```",
        "",
        sql
    )

    return sql.strip()

In [17]:
def generate_sql(question):

    prompt = f"""
You are an expert SQLite SQL generator.

Convert the user's natural-language question
into a SQLite SELECT query.

DATABASE SCHEMA:

{schema}

RULES:

1. Generate ONLY SELECT queries.
2. Use ONLY the sales table.
3. Use ONLY the columns provided in the schema.
4. Do NOT generate INSERT.
5. Do NOT generate UPDATE.
6. Do NOT generate DELETE.
7. Do NOT generate DROP.
8. Do NOT generate ALTER.
9. Do NOT generate CREATE.
10. Do NOT generate REPLACE.
11. Do NOT generate ATTACH.
12. Do NOT generate DETACH.
13. Use valid SQLite syntax.
14. Return ONLY the SQL query.
15. Do not provide explanations.
16. Do not use markdown.

USER QUESTION:

{question}
"""

    # Current Gemini model
    model_name = "gemini-3.6-flash"

    for attempt in range(3):

        try:
            print(f"\nTrying Gemini model: {model_name}")

            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )

            sql = response.text
            sql = clean_sql(sql)

            print("Gemini response received.")

            return sql

        except Exception as e:

            print(f"Attempt {attempt + 1} failed.")
            print("Error:", e)

            if attempt < 2:
                print("Retrying in 3 seconds...")
                time.sleep(3)

    raise Exception("Gemini API is currently unavailable.")

In [7]:
def execute_sql(sql):

    print("\nGENERATED SQL")
    print("-" * 60)
    print(sql)

    # Validate
    if not validate_sql(sql):
        print("\nUnsafe SQL rejected.")
        return

    # Execute
    try:
        cursor.execute(sql)
        results = cursor.fetchall()

        print("\nQUERY RESULT")
        print("-" * 60)

        if results:
            for row in results:
                print(row)
        else:
            print("No records found.")

    except sqlite3.Error as e:
        print("\nSQLite Error:")
        print(e)

In [21]:
print("=" * 60)
print("NATURAL LANGUAGE QUESTION")
print("=" * 60)

question = input("\nAsk a question about the sales data: ")

print("\nYour question:")
print(question)

try:

    generated_sql = generate_sql(question)

    execute_sql(generated_sql)

except Exception as e:

    print("\nGemini API Error:")
    print(e)

NATURAL LANGUAGE QUESTION

Ask a question about the sales data: show all completed sales

Your question:
show all completed sales

Trying Gemini model: gemini-3.6-flash
Gemini response received.

GENERATED SQL
------------------------------------------------------------
SELECT * FROM sales WHERE Status = 'Completed'

QUERY RESULT
------------------------------------------------------------
(1, 'Laptop', 'Electronics', 'Chennai', 2, 120000.0, '2026-01-10', 'Completed')
(2, 'Mobile', 'Electronics', 'Coimbatore', 3, 75000.0, '2026-01-12', 'Completed')
(3, 'Headphones', 'Accessories', 'Bengaluru', 5, 25000.0, '2026-01-15', 'Completed')
(5, 'Keyboard', 'Accessories', 'Madurai', 4, 12000.0, '2026-01-22', 'Completed')
(6, 'Monitor', 'Electronics', 'Coimbatore', 3, 45000.0, '2026-02-02', 'Completed')
(7, 'Mouse', 'Accessories', 'Chennai', 10, 10000.0, '2026-02-05', 'Completed')
(8, 'Printer', 'Electronics', 'Bengaluru', 2, 30000.0, '2026-02-10', 'Completed')
(9, 'Laptop', 'Electronics', 'Madur